In [ ]:
#全组合结果
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score
from sklearn.base import clone

# ===================== 完全对齐旧代码的核心参数 =====================
SEED = 42
CV_FOLDS = 5
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

# ===================== 1. 旧代码同款 PLS-DA =====================
class PLSDA(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=10):
        self.n_components = n_components
        self.pls = None
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        max_comp = min(X.shape[1], X.shape[0] - 1)
        n_comp = min(self.n_components, max_comp)
        self.pls = PLSRegression(n_components=n_comp)
        self.pls.fit(X, pd.get_dummies(y).values)
        return self

    def predict(self, X):
        pred = self.pls.predict(X)
        return self.classes_[np.argmax(pred, axis=1)]

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

# ===================== 2. 你指定的 已SG预处理 数据集 =====================
train_path = r"H:\图像标记\train_test\sg三类精准平衡后_训练集.xlsx"
test_path = r"H:\图像标记\train_test\sg610-0.52SAVI.xlsx"
save_dir = r"H:\图像标记\全组合波段"
save_path = os.path.join(save_dir, "全组合波段筛选结果4.xlsx")
# 自动创建保存文件夹
os.makedirs(save_dir, exist_ok=True)

# ===================== 3. 波段组合规则 =====================
fixed_bands = ['Band_5', 'Band_31', 'Band_35']
group1 = ['Band_40', 'Band_44', 'Band_49']
group2 = ['Band_73', 'Band_76']
group3 = ['Band_91', 'Band_93', 'Band_94', 'Band_95']

# 生成24种组合 + 全波段
all_combinations = []
for g1 in group1:
    for g2 in group2:
        for g3 in group3:
            combo = fixed_bands + [g1, g2, g3]
            all_combinations.append(combo)
all_bands = fixed_bands + group1 + group2 + group3
all_combinations.append(all_bands)

print(f"总组合数：{len(all_combinations)}")

# ===================== 4. 加载数据 =====================
train_df = pd.read_excel(train_path, engine='openpyxl')
test_df = pd.read_excel(test_path, engine='openpyxl')

X_train = train_df.iloc[:, 1:]  # 波段
y_train = train_df.iloc[:, 0]   # 标签
X_test = test_df.iloc[:, 1:]
y_test = test_df.iloc[:, 0]

# ===================== 5. 遍历组合训练 & 评估（完全对齐旧代码） =====================
results = []
base_model = PLSDA(n_components=10)

for i, bands in enumerate(all_combinations, 1):
    X_tr = X_train[bands].values
    X_te = X_test[bands].values

    # 1) 5折分层交叉验证（和旧代码完全一样）
    model_cv = clone(base_model)
    cv_acc = cross_val_score(model_cv, X_tr, y_train, cv=skf, scoring='accuracy').mean()

    # 2) 全训练集训练
    model_fit = clone(base_model)
    model_fit.fit(X_tr, y_train)
    y_pred = model_fit.predict(X_te)

    # 3) 各类 Recall（和旧代码计算方式完全一致）
    recall = []
    for cls in [0, 1, 2]:
        mask = y_test == cls
        if np.sum(mask) > 0:
            r = accuracy_score(y_test[mask], y_pred[mask])
            recall.append(r)
        else:
            recall.append(0.0)
    recall_0, recall_1, recall_2 = recall
    avg_recall = np.mean(recall)

    combo_name = f"组合{i}" if i <= 24 else "全12波段"

    results.append({
        "组合编号": combo_name,
        "使用波段": ", ".join(bands),
        "训练集5折CV准确率": round(cv_acc, 4),
        "水稻(0)Recall": round(recall_0, 4),
        "稗草(1)Recall": round(recall_1, 4),
        "千金子(2)Recall": round(recall_2, 4),
        "测试集平均Recall": round(avg_recall, 4)
    })

    print(f"{combo_name} | CV:{cv_acc:.2%} | 平均Recall:{avg_recall:.2%}")

# ===================== 保存结果 =====================
result_df = pd.DataFrame(results)
result_df.to_excel(save_path, index=False)
print(f"\n✅ 全部完成！结果已保存到：\n{save_path}")

In [ ]:
#训练集精度排名前五的组合统计频次
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import os

# =========================================================
# 读取数据
# =========================================================
file_path = r"H:\图像标记\全组合波段\交叉验证前5.xlsx"

df = pd.read_excel(file_path, header=None)

# 第一列：波段
# 第二列：频数
bands = df.iloc[:, 0].astype(str)
freqs = df.iloc[:, 1].astype(float)

# =========================================================
# 提取波段编号
# =========================================================
band_numbers = bands.str.extract(r'(\d+)').astype(int)[0]

# =========================================================
# 三个组合
# =========================================================
group1 = [40, 44, 49]
group2 = [73, 76]
group3 = [91, 93, 94, 95]

# =========================================================
# 自定义x坐标
# 同组间距一致
# 不同组之间更宽
# =========================================================
x_positions = []

# 第一组
x_positions.extend([1, 2, 3])

# 第二组（留更宽间隔）
x_positions.extend([5.5, 6.5])

# 第三组（再留更宽间隔）
x_positions.extend([9, 10, 11, 12])

# =========================================================
# 颜色设置
# =========================================================
colors = []

for b in band_numbers:

    # group1 绿色
    if b in group1:
        colors.append('#44CFC6')

    # group2 粉红色
    elif b in group2:
        colors.append('#DC6D76')

    # group3 粉红色
    elif b in group3:
        colors.append('#DC6D76')

    # 其他（保险）
    else:
        colors.append('#BFBFBF')

# =========================================================
# 创建画布
# =========================================================
plt.figure(figsize=(12, 12))

# =========================================================
# 棒棒糖竖线
# =========================================================
for x, y, c in zip(x_positions, freqs, colors):

    plt.vlines(
        x=x,
        ymin=0,
        ymax=y,
        color=c,
        linewidth=2.5,
        alpha=0.95
    )

# =========================================================
# 顶部圆点（无轮廓）
# =========================================================
plt.scatter(
    x_positions,
    freqs,
    s=220,
    c=colors,
    edgecolors='none',
    zorder=3
)

# =========================================================
# 0频数增强显示
# 防止贴在x轴看不见
# =========================================================
for i in range(len(freqs)):

    if freqs.iloc[i] == 0:

        plt.scatter(
            x_positions[i],
            0.03,
            s=220,
            color=colors[i],
            edgecolors='none',
            zorder=4
        )

# =========================================================
# 坐标轴标签
# =========================================================
plt.xlabel(
    'Band Number',
    fontsize=18,
    fontweight='bold'
)

plt.ylabel(
    'Frequency',
    fontsize=18,
    fontweight='bold'
)

# =========================================================
# x轴刻度
# 去掉 Band_ 前缀
# =========================================================

# 你的原有代码
plt.xticks(
    x_positions,    
    [str(i) for i in band_numbers],
    fontsize=13
)

# ==========关键：y轴强制只使用整数刻度==========
ax = plt.gca()
ax.yaxis.set_major_locator(MaxNLocator(integer=True)) 

plt.yticks(fontsize=13)


# =========================================================
# SCI风格边框
# =========================================================
ax = plt.gca()

for spine in ax.spines.values():
    spine.set_linewidth(1.2)
    spine.set_color('#000000')

# =========================================================
# 去网格
# =========================================================
plt.grid(False)

# =========================================================
# 留白优化
# =========================================================
plt.tight_layout()

# =========================================================
# 保存路径
# =========================================================
save_dir = r"H:\图像标记\全组合波段"

os.makedirs(save_dir, exist_ok=True)

# PNG
plt.savefig(
    os.path.join(save_dir, 'SCI_Lollipop49.png'),
    dpi=600,
    bbox_inches='tight'
)

# TIFF
plt.savefig(
    os.path.join(save_dir, 'SCI_Lollipop49.tiff'),
    dpi=600,
    bbox_inches='tight'
)

# SVG（SCI推荐）
plt.savefig(
    os.path.join(save_dir, 'SCI_Lollipop49.svg'),
    bbox_inches='tight'
)

# =========================================================
# 显示图像
# =========================================================
plt.show()

print("SCI棒棒糖图已保存完成！")